In [ ]:

%pip install -q -U \
    transformers \
    accelerate \
    bitsandbytes \
    langchain \
    langchain-core \
    langchain-huggingface \
    langchain-community \
    sentence-transformers \
    faiss-cpu \
    pandas \
    numpy \
    scipy \
    scikit-learn \
    openpyxl \
    pydantic \
    plotly \
    streamlit \
    pyngrok


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.5.4 which is incompatible.


In [ ]:

%pip install --no-cache-dir --force-reinstall "numpy==2.2.6"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 51.6 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.4
    Uninstalling numpy-2.5.4:
      Successfully uninstalled numpy-2.5.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.


In [ ]:

%pip install -q fastapi uvicorn nest-asyncio


In [ ]:

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16
)

model.eval()

def ask_qwen(prompt, max_new_tokens=250):
    messages = [
        {
            "role": "system",
            "content": (
                "You are Intellianalyzer, a concise and accurate "
                "data analysis assistant. Never invent statistics."
            )
        },
        {"role": "user", "content": prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()



[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [ ]:

import pandas as pd
import numpy as np
import json
from pathlib import Path

def load_dataset(file):
    name = file.name.lower() if hasattr(file, "name") else str(file).lower()

    if name.endswith(".csv"):
        return pd.read_csv(file)

    if name.endswith((".xlsx", ".xls")):
        return pd.read_excel(file)

    raise ValueError("Please upload a CSV or Excel file.")


def profile_dataset(df):
    if df.empty:
        raise ValueError("The uploaded dataset contains no rows.")

    if len(df.columns) == 0:
        raise ValueError("The dataset contains no columns.")

    # Avoid ambiguous duplicate column names in downstream analysis.
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    if df.columns.duplicated().any():
        raise ValueError(
            "Duplicate column names found. Rename them before analysis."
        )

    numeric = df.select_dtypes(include=np.number)
    categorical = df.select_dtypes(
        include=["object", "category", "bool", "string"]
    )

    report = {
        "dataset": {
            "rows": int(len(df)),
            "columns": int(len(df.columns)),
            "duplicate_rows": int(df.duplicated().sum()),
            "memory_mb": round(
                df.memory_usage(deep=True).sum() / (1024**2), 2
            ),
        },
        "columns": {},
        "correlations": {},
        "warnings": []
    }

    for col in df.columns:
        series = df[col]
        info = {
            "dtype": str(series.dtype),
            "missing_count": int(series.isna().sum()),
            "missing_percent": round(
                float(series.isna().mean() * 100), 2
            ),
            "unique_values": int(series.nunique(dropna=True))
        }

        if col in numeric.columns:
            values = series.replace([np.inf, -np.inf], np.nan).dropna()

            info["statistics"] = {
                "mean": float(values.mean()) if len(values) else None,
                "median": float(values.median()) if len(values) else None,
                "std": float(values.std()) if len(values) > 1 else None,
                "min": float(values.min()) if len(values) else None,
                "max": float(values.max()) if len(values) else None,
                "q25": float(values.quantile(0.25)) if len(values) else None,
                "q75": float(values.quantile(0.75)) if len(values) else None
            }

            if len(values) and values.nunique() <= 1:
                report["warnings"].append(
                    f"{col} is constant or has only one valid value."
                )

        else:
            # Limit category summaries to the 10 most frequent values.
            counts = series.value_counts(dropna=True).head(10)
            info["top_values"] = {
                str(k): int(v) for k, v in counts.items()
            }

        report["columns"][str(col)] = info

    if len(numeric.columns) >= 2:
        corr = numeric.corr().replace([np.inf, -np.inf], np.nan)

        for i, col1 in enumerate(corr.columns):
            for col2 in corr.columns[i + 1:]:
                value = corr.loc[col1, col2]
                if pd.notna(value):
                    report["correlations"][f"{col1} vs {col2}"] = round(
                        float(value), 4
                    )

    for col in df.columns:
        if df[col].isna().any():
            report["warnings"].append(
                f"{col} contains missing values."
            )

    if report["dataset"]["duplicate_rows"]:
        report["warnings"].append(
            "Duplicate rows were detected; they were not automatically removed."
        )

    return df, report


def report_to_text(report):
    return json.dumps(report, indent=2, ensure_ascii=False)


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


In [ ]:

from pydantic import BaseModel, Field
from typing import List, Optional
from langchain_core.output_parsers import PydanticOutputParser


class Hypothesis(BaseModel):
    title: str = Field(description="Short hypothesis title")
    question: str = Field(description="Question to investigate")
    null_hypothesis: str = Field(description="The statistical null hypothesis")
    alternative_hypothesis: str = Field(
        description="The statistical alternative hypothesis"
    )
    variables: List[str] = Field(
        description="Exact column names from the dataset"
    )
    suggested_test: str = Field(
        description=(
            "One of: mann_whitney_u, t_test, chi_square, "
            "spearman, pearson, kruskal_wallis, descriptive_only"
        )
    )
    rationale: str = Field(description="Why this hypothesis is useful")
    limitations: str = Field(
        description="Important assumptions or limitations"
    )


class AnalysisPlan(BaseModel):
    dataset_summary: str
    data_quality_findings: List[str]
    recommended_questions: List[str]
    hypotheses: List[Hypothesis]


parser = PydanticOutputParser(pydantic_object=AnalysisPlan)



In [ ]:

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

analysis_prompt = PromptTemplate(
    template="""
You are the analysis planner for Intellianalyzer.

Analyze the dataset profile below and produce a useful, realistic
exploratory analysis plan.

Rules:
- Use only column names that exist in the profile.
- Do not invent sample sizes, p-values, test statistics, or results.
- Suggest hypotheses that can reasonably be investigated.
- Choose descriptive_only when no appropriate test is apparent.
- Select no more than 5 hypotheses.
- For numerical variables, consider their distribution and groups.
- For categorical variables, consider association tests.
- Distinguish correlation from causation.
- Mention relevant assumptions and limitations.
- Return valid structured output matching the schema.

DATASET PROFILE:
{profile}

{format_instructions}
""",
    input_variables=["profile"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }
)


def generate_analysis_plan(report):
    prompt = analysis_prompt.format(
        profile=report_to_text(report)
    )

    raw_output = ask_qwen(prompt, max_new_tokens=1400)

    # Validate the generated JSON using Pydantic.
    # If validation fails, show the raw response for debugging.
    try:
        return parser.parse(raw_output)
    except Exception as exc:
        print("Could not parse Qwen's response:", exc)
        print("Raw model response:\n", raw_output)
        raise




In [ ]:

from scipy import stats
from scipy.stats import chi2_contingency
from itertools import combinations


def run_hypothesis_test(df, hypothesis):
    h = hypothesis.model_dump() if hasattr(
        hypothesis, "model_dump"
    ) else hypothesis

    columns = h["variables"]
    test_name = h["suggested_test"]

    if any(c not in df.columns for c in columns):
        return {"status": "rejected", "reason": "Unknown column name"}

    if test_name == "descriptive_only":
        return {
            "status": "not_tested",
            "reason": "This hypothesis requires descriptive analysis."
        }

    try:
        if test_name in ("pearson", "spearman"):
            if len(columns) != 2:
                raise ValueError("Correlation requires exactly 2 columns.")

            x = pd.to_numeric(df[columns[0]], errors="coerce")
            y = pd.to_numeric(df[columns[1]], errors="coerce")
            valid = pd.DataFrame({"x": x, "y": y}).replace(
                [np.inf, -np.inf], np.nan
            ).dropna()

            if len(valid) < 3:
                raise ValueError("Not enough valid observations.")
            if valid["x"].nunique() < 2 or valid["y"].nunique() < 2:
                raise ValueError("A constant variable cannot be correlated.")

            fn = stats.pearsonr if test_name == "pearson" else stats.spearmanr
            result = fn(valid["x"], valid["y"])
            statistic, p_value = float(result.statistic), float(result.pvalue)
            n = len(valid)

        elif test_name in ("mann_whitney_u", "t_test"):
            if len(columns) != 2:
                raise ValueError(
                    "This test requires one numeric and one grouping column."
                )

            a, b = columns
            numeric_col = a if pd.api.types.is_numeric_dtype(df[a]) else b
            group_col = b if numeric_col == a else a

            if not pd.api.types.is_numeric_dtype(df[numeric_col]):
                raise ValueError("A numeric variable is required.")

            clean = df[[numeric_col, group_col]].dropna()
            groups = clean[group_col].unique()

            if len(groups) != 2:
                raise ValueError("Exactly two groups are required.")

            x = pd.to_numeric(
                clean.loc[clean[group_col] == groups[0], numeric_col],
                errors="coerce"
            ).dropna()
            y = pd.to_numeric(
                clean.loc[clean[group_col] == groups[1], numeric_col],
                errors="coerce"
            ).dropna()

            if min(len(x), len(y)) < 2:
                raise ValueError("Each group needs at least 2 valid observations.")

            if test_name == "mann_whitney_u":
                result = stats.mannwhitneyu(
                    x, y, alternative="two-sided"
                )
            else:
                result = stats.ttest_ind(
                    x, y, equal_var=False, nan_policy="omit"
                )

            statistic, p_value = float(result.statistic), float(result.pvalue)
            n = len(x) + len(y)

        elif test_name == "chi_square":
            if len(columns) != 2:
                raise ValueError("Chi-square requires exactly 2 columns.")

            table = pd.crosstab(df[columns[0]], df[columns[1]])
            if table.shape[0] < 2 or table.shape[1] < 2:
                raise ValueError("Both columns must have at least 2 categories.")

            statistic, p_value, dof, expected = chi2_contingency(table)
            statistic, p_value = float(statistic), float(p_value)
            n = int(table.to_numpy().sum())

        elif test_name == "kruskal_wallis":
            if len(columns) != 2:
                raise ValueError(
                    "Kruskal-Wallis requires a numeric variable and a group."
                )

            a, b = columns
            numeric_col = a if pd.api.types.is_numeric_dtype(df[a]) else b
            group_col = b if numeric_col == a else a

            if not pd.api.types.is_numeric_dtype(df[numeric_col]):
                raise ValueError("A numeric variable is required.")

            clean = df[[numeric_col, group_col]].dropna()
            groups = [
                pd.to_numeric(
                    clean.loc[clean[group_col] == g, numeric_col],
                    errors="coerce"
                ).dropna().to_numpy()
                for g in clean[group_col].unique()
            ]
            groups = [g for g in groups if len(g) > 0]

            if len(groups) < 2:
                raise ValueError("At least 2 non-empty groups are required.")

            result = stats.kruskal(*groups)
            statistic, p_value = float(result.statistic), float(result.pvalue)
            n = sum(map(len, groups))

        else:
            raise ValueError(f"Unsupported test: {test_name}")

        return {
            "status": "success",
            "test": test_name,
            "variables": columns,
            "sample_size": int(n),
            "statistic": statistic,
            "p_value": p_value,
            "significant_at_0_05": bool(p_value < 0.05),
            "interpretation_note": (
                "Statistical significance does not establish causation. "
                "Check test assumptions and consider multiple-testing correction."
            )
        }

    except Exception as exc:
        return {
            "status": "error",
            "test": test_name,
            "variables": columns,
            "reason": str(exc)
        }



In [ ]:

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)


def build_knowledge_base(report, plan, results):
    documents = []

    documents.append(Document(
        page_content="DATASET PROFILE\n" + report_to_text(report),
        metadata={"source": "dataset_profile"}
    ))

    for h in plan.hypotheses:
        documents.append(Document(
            page_content=(
                "HYPOTHESIS\n"
                + h.model_dump_json(indent=2)
            ),
            metadata={"source": "hypothesis", "title": h.title}
        ))

    for result in results:
        documents.append(Document(
            page_content=(
                "STATISTICAL TEST RESULT\n"
                + json.dumps(result, indent=2, default=str)
            ),
            metadata={
                "source": "statistical_result",
                "title": result["hypothesis"]
            }
        ))

    return FAISS.from_documents(documents, embedding_model)



/tmp/ipykernel_20270/3774046469.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:

def ask_dataset(question, k=4):
    retrieved_docs = vector_store.similarity_search(
        question, k=k
    )

    context = "\n\n".join(
        f"[Source: {doc.metadata.get('source', 'analysis')}]\n"
        f"{doc.page_content}"
        for doc in retrieved_docs
    )

    prompt = f"""
You are Intellianalyzer's dataset question-answering assistant.

Answer the user's question using the retrieved analysis below.

Rules:
- Use only evidence supported by the context.
- Never invent statistics or claim an untested hypothesis was verified.
- Distinguish statistical association from causation.
- If evidence is missing, say what information is missing.
- Cite the relevant source type in your answer.
- Be concise and helpful.

RETRIEVED ANALYSIS:
{context}

USER QUESTION:
{question}

ANSWER:
"""

    answer = ask_qwen(prompt, max_new_tokens=450)

    return {
        "answer": answer,
        "sources": [
            {
                "source": doc.metadata.get("source"),
                "title": doc.metadata.get("title", "")
            }
            for doc in retrieved_docs
        ]
    }



In [14]:

import io
import pandas as pd
import numpy as np
import threading
import time

from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from scipy import stats
import uvicorn

api = FastAPI(title="Intellianalyzer API", version="1.0.0")

api.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # Development only
    allow_methods=["*"],
    allow_headers=["*"],
)

# Store the latest analysis for the chat endpoint.
app_state = {
    "ready": False,
    "filename": None,
    "df": None,
    "report": None,
    "plan": None,
    "results": None,
    "vector_store": None,
}


@api.get("/health")
def health():
    return {
        "status": "ok",
        "model_loaded": model is not None,
        "dataset_analyzed": app_state["ready"],
    }


@api.post("/analyze")
async def analyze_dataset(file: UploadFile = File(...)):
    filename = file.filename or "dataset.csv"
    extension = filename.lower()

    if not extension.endswith((".csv", ".xlsx", ".xls")):
        raise HTTPException(
            status_code=400,
            detail="Upload a CSV or Excel file."
        )

    content = await file.read()

    if not content:
        raise HTTPException(status_code=400, detail="The file is empty.")

    # Basic upload limit for the MVP.
    if len(content) > 25 * 1024 * 1024:
        raise HTTPException(
            status_code=413,
            detail="Maximum upload size is 25 MB."
        )

    try:
        buffer = io.BytesIO(content)

        if extension.endswith(".csv"):
            dataframe = pd.read_csv(buffer)
        else:
            dataframe = pd.read_excel(buffer)

        dataframe, report = profile_dataset(dataframe)
        plan = generate_analysis_plan(report)

        results = []
        for hypothesis in plan.hypotheses:
            result = run_hypothesis_test(dataframe, hypothesis)
            results.append({
                "hypothesis": hypothesis.title,
                "question": hypothesis.question,
                "result": result,
            })

        # Build a fresh knowledge base for this dataset.
        store = build_knowledge_base(report, plan, results)

        # Replace the previous analysis only after successful processing.
        app_state.update({
            "ready": True,
            "filename": filename,
            "df": dataframe,
            "report": report,
            "plan": plan,
            "results": results,
            "vector_store": store,
        })

        return {
            "filename": filename,
            "profile": report,
            "analysis_plan": plan.model_dump(),
            "test_results": results,
        }

    except HTTPException:
        raise
    except Exception as exc:
        raise HTTPException(
            status_code=500,
            detail=f"Analysis failed: {type(exc).__name__}: {str(exc)[:500]}"
        )


class ChatRequest(BaseModel):
    question: str


@api.post("/chat")
def chat(request: ChatRequest):
    if not app_state["ready"]:
        raise HTTPException(
            status_code=400,
            detail="Analyze a dataset before asking questions."
        )

    question = request.question.strip()

    if not question:
        raise HTTPException(
            status_code=400,
            detail="Please enter a question."
        )

    if len(question) > 2000:
        raise HTTPException(
            status_code=400,
            detail="Question is too long (maximum 2000 characters)."
        )

    try:
        documents = app_state["vector_store"].similarity_search(
            question, k=4
        )

        context = "\n\n".join(
            f"[Source: {doc.metadata.get('source', 'analysis')}]\n"
            f"{doc.page_content}"
            for doc in documents
        )

        prompt = f"""
You are Intellianalyzer, a data analysis assistant.

Answer using the retrieved evidence. Never invent statistics.
Distinguish correlation from causation. If evidence is insufficient,
say so clearly. Be concise and practical.

RETRIEVED EVIDENCE:
{context}

QUESTION:
{question}

ANSWER:
"""

        answer = ask_qwen(prompt, max_new_tokens=450)

        return {
            "answer": answer,
            "sources": [
                {
                    "source": doc.metadata.get("source"),
                    "title": doc.metadata.get("title", ""),
                }
                for doc in documents
            ],
        }

    except Exception as exc:
        raise HTTPException(
            status_code=500,
            detail=f"Chat failed: {type(exc).__name__}: {str(exc)[:300]}"
        )


print("FastAPI application created.")


FastAPI application created.


In [ ]:
from getpass import getpass
from pyngrok import ngrok, exception
import nest_asyncio
import requests

nest_asyncio.apply()

NGROK_TOKEN = "your_ngrok_auth_token"
try:
    ngrok.set_auth_token(NGROK_TOKEN)
except exception.PyngrokNgrokError:
    # Ignore the error if the authtoken is already configured
    pass

# Close previous tunnels if this cell is rerun.
ngrok.kill()

def run_api():
    uvicorn.run(
        api,
        host="0.0.0.0",
        port=8000,
        log_level="warning",
    )

api_thread = threading.Thread(
    target=run_api,
    daemon=True
)
api_thread.start()

# Wait for FastAPI to start.
for attempt in range(30):
    try:
        response = requests.get(
            "http://127.0.0.1:8000/health",
            timeout=2
        )
        if response.ok:
            break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError("FastAPI failed to start on port 8000.")

tunnel = ngrok.connect(8000, "http")
API_URL = tunnel.public_url.rstrip("/")

print("API URL:", API_URL)
print("Health check:", f"{API_URL}/health")
print("API documentation:", f"{API_URL}/docs")

API URL: https://amplify-ivy-poster.ngrok-free.dev
Health check: https://amplify-ivy-poster.ngrok-free.dev/health
API documentation: https://amplify-ivy-poster.ngrok-free.dev/docs
